# Exercise 4: Participial clauses

*The Computational Phenotype · version 0.1.0*

> Translate *Dün şehre gelen adam* and *Ali'nin okuduğu kitap*. Identify the participle in each, say whether the head noun is the subject or a non-subject of the relative clause, and sketch the dependency structure.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" pandas`. Download the language resources once with `turkicnlp.download(lang, processors=[...])`, as shown in the first code cell. Cells that fetch data from the web need internet access.

In [ ]:
import hfst
import turkicnlp
from turkicnlp.resources.registry import ModelRegistry

turkicnlp.download("tur", processors=["morph"])          # Apertium Turkish transducers (once)
tur_dir = ModelRegistry.get_model_path("tur", "morph", "apertium", script="Latn")

def load(path):
    stream = hfst.HfstInputStream(str(path)); fst = stream.read(); stream.close()
    fst.convert(hfst.ImplementationType.HFST_OLW_TYPE)   # optimized lookup: no epsilon symbols in output
    return fst

analyser = load(tur_dir / "tur.automorf.hfst")
generator = load(tur_dir / "tur.autogen.hfst")
analyse = lambda word: sorted({a for a, _ in analyser.lookup(word)})
generate = lambda analysis: sorted({f for f, _ in generator.lookup(analysis)})

In [ ]:
for word in ["gelen", "okuduğu"]:
    print(word, analyse(word))

In [ ]:
turkicnlp.download("tur", processors=["tokenize", "pos", "lemma", "depparse"])
nlp = turkicnlp.Pipeline("tur", processors=["tokenize", "pos", "lemma", "depparse"])
# End each phrase with a full stop: without it, the 0.5.0 tokenizer can split the last word.
for phrase in ["Dün şehre gelen adam.", "Ali'nin okuduğu kitap."]:
    print(phrase)
    for w in nlp(phrase).words:
        print(f"   {w.id} {w.text:<9} {w.upos:<6} head={w.head} {w.deprel}")

## Worked solution

**Dün şehre gelen adam** 'the man who came to the city yesterday'. The participle is *gelen* (*gel-en*, come-PTCP, Apertium tag `gpr_rsub`, a subject participle). The head noun *adam* is the **subject** of the relative clause: the man came. *Dün* 'yesterday' and *şehre* 'to the city' are dependents of *gelen*.

**Ali'nin okuduğu kitap** 'the book that Ali read'. The participle is *okuduğu* (*oku-duğ-u*, read-NMLZ-POSS.3SG). The head noun *kitap* is a **non-subject** (the object) of the relative clause; the clause's subject *Ali'nin* is in the genitive, and the possessive *-u* on the participle agrees with it.

**Dependency sketch** (UD basic relations; arrows go from head to dependent):

```
adam ──acl──▶ gelen ──obl──▶ şehre          kitap ──acl──▶ okuduğu ──nsubj──▶ Ali'nin
                    └─advmod/obl─▶ Dün
```

The parser output reproduces this structure: *gelen* → *adam* (`acl`), *şehre* and *Dün* → *gelen* (`obl`; the parser tags *Dün* as a noun, where an adverb analysis with `advmod` would also be defensible), and *Ali'nin* → *okuduğu* (`nsubj`) → *kitap* (`acl`). Note that the participle stands directly before its head noun; the longer arcs run from the clause's own dependents to the participle.